In [37]:
import random
import os
import time
from typing import Any
from abc import ABC, abstractmethod
import random
import time
from typing import Any


class BaseModel(ABC):
    def __init__(self, model_name: str, **kwargs):
        self._name = model_name
        self.max_tokens = kwargs.get('max_tokens', 512)
        self.temperature = kwargs.get('temperature', 0.7)
        self.top_p = kwargs.get('top_p', 0.9)
        self.reasoning_effort = kwargs.get('reasoning_effort', None)
        self.n = kwargs.get('n', 1)
        self.input_tokens = 0
        self.output_tokens = 0

    @abstractmethod
    def generate(self, messages) -> str:
        pass

    @property
    def name(self) -> str:
        return self._name
    
    def from_text_to_tokens(self, text: str) -> list[int]:
        """Convert text to tokens."""
        raise NotImplementedError("This method should be implemented by subclasses.")
    
    def from_token_to_text(self, token: int) -> str:
        """Convert a token ID back to text."""
        raise NotImplementedError("This method should be implemented by subclasses.")




In [ ]:
from together import Together
import together
TogetherAIClient = Together(api_key="__YOUR_API_KEY__")

class TogetherAIModel(BaseModel):
    def __init__(self, model_name: str, **kwargs):
        super().__init__(model_name=model_name, **kwargs)

    def retry_with_exponential_backoff(  # type: ignore
        func,
        initial_delay: float = 1,
        exponential_base: float = 2,
        jitter: bool = True,
        max_retries: int = 5,
    ):
        """Retry a function with exponential backoff."""

        def wrapper(*args, **kwargs):  # type: ignore
            # Initialize variables
            num_retries = 0
            delay = initial_delay

            # Loop until a successful response or max_retries is hit or an exception is raised
            while True:
                try:
                    return func(*args, **kwargs)
                except together.error.InvalidRequestError as e:
                    raise e
                except Exception as e:
                    num_retries += 1
                    delay *= exponential_base * (1 + jitter * random.random())
                    print(f"#{num_retries} Error occurred: {e}.\n Retrying in {delay} seconds.")
                    # Sleep for the delay
                    time.sleep(delay)

        return wrapper

    @retry_with_exponential_backoff
    def generate(self, messages) -> str:
        """
        Chat completion using the chat/completions endpoint.
        Supports multi-modal inputs (text + images) for vision models.
        """
        response = TogetherAIClient.chat.completions.create(
            model=self.name,
            messages=messages,
            # max_tokens=self.max_tokens,
            max_new_tokens=1024,
            temperature=self.temperature,
            top_p=self.top_p,
            n=self.n,
        )
        
        usage = getattr(response, "usage", None)
        if usage:
            self.input_tokens += usage.prompt_tokens
            self.output_tokens += usage.completion_tokens
            print(f"Total input tokens: {self.input_tokens}, Total output tokens: {self.output_tokens}")

        # Raise OpenRouterError if we get invalid response to trigger retry
        if not response or not hasattr(response, 'choices') or not response.choices:
            raise ValueError("Zero response from Together API")

        predictions = [choice.message.content.strip(
        ) for choice in response.choices if choice.message.content.strip()]

        
        return predictions[0]

In [ ]:
import os
import requests
API_KEY = "__YOUR_API_KEY__"  # Replace with your actual API key or set as environment variable
MODEL = "gemini-2.0-flash"
URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

def gemini_prompt(prompt: str) -> str:
    """Send a prompt to Gemini API and return the generated text."""
    if not API_KEY:
        raise ValueError("Missing GEMINI_API_KEY. Please set it as an environment variable.")

    headers = {
        "Content-Type": "application/json",
        "X-goog-api-key": API_KEY,
    }

    payload = {
        "contents": [
            {
                "parts": [{"text": prompt}]
            }
        ]
    }

    resp = requests.post(URL, headers=headers, json=payload, timeout=30)
    resp.raise_for_status()

    data = resp.json()
    try:
        return data["candidates"][0]["content"]["parts"][0]["text"]
    except Exception:
        return str(data)  # fallback: dump raw response if unexpected

In [40]:
#model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-70B-free"
model_name ="lgai/exaone-deep-32b"
#model_name = "lgai/exaone-3-5-32b-instruct"

In [41]:
agent = TogetherAIModel(model_name)

In [42]:
import ast
import pandas as pd
import re
import re

# Function to remove Python code blocks from a string
def remove_python_block(input_string: str) -> str:
    return re.sub(r'```python.*?```', '', input_string, flags=re.DOTALL)

def convert_csv_to_json(csv_file):
    # Read CSV file into a DataFrame
    df = pd.read_csv(csv_file, encoding='utf-8')
    df['test_list'] = df['test_list'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    # Remove "Example:" and anything following it in the 'instruction_en' column
    df['instruction_en'] = df['instruction_en'].str.replace(r'\s*Example:.*', '', regex=True)

    # Convert DataFrame to a dictionary and return
    return df.to_dict(orient='records')

In [43]:
import signal

# Timeout handler
def _timeout_handler(signum, frame):
    raise TimeoutError("Execution timed out")

def evaluate_solution(solution_code: str, unit_tests: list[str], timeout_per_test: int = 5) -> int:
    # Clean solution code (optional: remove markdown fences)
    solution_code = solution_code.strip('` \n').replace('python\n', '').strip()
    
    # Prepare namespace
    namespace = {}
    
    # Execute solution code with timeout
    try:
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(timeout_per_test * len(unit_tests))  # total timeout
        exec(solution_code, namespace)
    except TimeoutError:
        print("⏱️ Timeout in solution code execution (skipping)")
    except Exception as e:
        print(f"❌ Error in solution code: {e}")
        return 0
    finally:
        signal.alarm(0)  # always cancel alarm
    
    # Evaluate unit tests
    passed_count = 0
    for i, test_stmt in enumerate(unit_tests):
        try:
            signal.alarm(timeout_per_test)
            exec(test_stmt, namespace)
            passed_count += 1
        except TimeoutError:
            print(f"⏱️ Test {i+1} timed out (skipping)")
        except AssertionError:
            print(f"❌ Test {i+1} failed: {test_stmt}")
        except SystemExit as e:
            print(f"⚠️ SystemExit in test {i+1}: {e.code}")
        except Exception as e:
            print(f"⚠️ Exception in test {i+1}: {e}")
        finally:
            signal.alarm(0)  # cancel any pending alarm

    return passed_count


In [44]:
def run_code(code: str):
    namespace = {}
    try:
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(60)  # total timeout for code + tests
        exec(code, namespace)
        signal.alarm(0)
    except TimeoutError:
        raise TimeoutError("Execution timed out")
    except AssertionError as e:
        raise AssertionError(f"Assertion failed: {e}")
    except SyntaxError as e:
        raise SyntaxError(f"Syntax error in code: {e}")
    except Exception as e:
        raise RuntimeError(f"Error while executing code: {repr(e)}")
    except SystemExit as e:
        raise RuntimeError(f"SystemExit occurred: {e.code}")
    except Exception as e:
        raise RuntimeError(f"Error while executing code: {repr(e)}")

In [45]:
def get_fix_instructions(error: Exception) -> str:
    if isinstance(error, TimeoutError):
        return (
            "⏱️ TimeoutError: The code took too long to finish running.\n"
            "- The program might be stuck in a loop or taking too long to process.\n"
            "- Check if there is a loop that doesn’t stop.\n"
            "- Test the code with smaller or simpler input data.\n"
            "- Make sure you're not doing unnecessary repeated calculations."
        )

    elif isinstance(error, AssertionError):
        return (
            f"🧪 AssertionError: {error}\n"
            "- The result of your code did not match what was expected.\n"
            "- Double-check the assertion conditions to ensure they are correct.\n"
            "- Carefully review your assertions and verify what they are expecting.\n"
            "- Carefully review your code logic to make sure it does what the test expects.\n"
            "- Make sure the values you're comparing are what you actually intended."
            "- Make sure the function name  is same as the unit test assertion function name ."
        )

    elif isinstance(error, SyntaxError):
        return (
            f"✏️ SyntaxError: {error}\n"
            "- There is a problem with how the code is written.\n"
            "- Check for missing colons `:`, parentheses `()`, or indentation.\n"
            "- Make sure strings are closed properly with matching quotes.\n"
            "- Review the line and nearby lines for typos or misplaced symbols."
        )

    elif isinstance(error, SystemExit):
        return (
            f"🚪 SystemExit: The program exited with code {error.code}.\n"
            "- The code called `exit()` or something that stops the program.\n"
            "- Only use exit calls if the program is supposed to stop.\n"
            "- If you don’t want the program to exit early, remove or comment out those lines."
        )

    elif isinstance(error, RuntimeError):
        return (
            f"🚨 RuntimeError: {repr(error)}\n"
            "- A general problem happened while the program was running.\n"
            "- Check the values being used in the part of the code that caused the error.\n"
            "- Make sure everything used has been defined correctly.\n"
            "- Ensure the logic and data flow make sense and follow the correct order."
        )

    else:
        return (
            f"❗ Unhandled Error: {repr(error)}\n"
            "- An unexpected issue occurred.\n"
            "- Review the error message to understand what part of the code is causing it.\n"
            "- Go over the code structure and logic step by step.\n"
            "- Make sure all variables and functions are used correctly."
        )


# Prompts

In [46]:
from tqdm import tqdm
import re
from IPython.display import clear_output
from pathlib import Path
import json

dev_set = convert_csv_to_json("test_v1_en_gemini.csv")

responses = []

EXAMPLES = '''
>> Example 1:
> Instruction
```python
def smallest_multiple(n):
    """প্রথম n সংখ্যার ক্ষুদ্রতম গুণিতক খুঁজে বের করার জন্য একটি ফাংশন লিখুন।"""
    """Translated: Write a function to find the smallest multiple of the first n numbers."""
    """Test Case : assert smallest_multiple(13) == 360360"""
```
> Solution
```python
def smallest_multiple(n):
    if (n<=2):
        return n
    i = n * 2
    factors = [number  for number in range(n, 1, -1) if number * 2 > n]
    while True:
        for a in factors:
            if i % a != 0:
                i += n
                break
            if (a == factors[-1] and i % a == 0):
                return i
                
def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"
    
def main():
    check(1, smallest_multiple(13), 360360)
    check(2, smallest_multiple(2), 2)
    check(3, smallest_multiple(1), 1)
```

>> Example 2:
> Instruction
```python
def add_dict(d1,d2):
    """সাধারণ কীগুলির জন্য মান যোগ করে দুটি অভিধানকে একত্রিত করার জন্য একটি ফাংশন লিখুন।"""
    """Translated: Write a function to merge two dictionaries by adding the values for common keys."""
    """Test Case : assert add_dict({{'a': 100, 'b': 200, 'c':300}},{{'a': 300, 'b': 200, 'd':400}}) == ({{'b': 400, 'd': 400, 'a': 400, 'c': 300}})"""
```
> Solution
```python
from collections import Counter

def add_dict(d1,d2):
    add_dict = Counter(d1) + Counter(d2)
    return add_dict  

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"
    
def main():
    check(1, add_dict({{'a': 100, 'b': 200, 'c':300}},{{'a': 300, 'b': 200, 'd':400}}), ({{'b': 400, 'd': 400, 'a': 400, 'c': 300}}))
    check(2, add_dict({{'a': 500, 'b': 700, 'c':900}},{{'a': 500, 'b': 600, 'd':900}}), ({{'b': 1300, 'd': 900, 'a': 1000, 'c': 900}}))
    check(3, add_dict({{'a':900,'b':900,'d':900}},{{'a':900,'b':900,'d':900}}), ({{'b': 1800, 'd': 1800, 'a': 1800}}))
```

>> Example 3:
> Instruction
```python
def count_Unset_Bits(n):
    """১ থেকে এন পর্যন্ত মোট আনসেট বিট গণনা করার জন্য একটি পাইথন ফাংশন লিখুন।"""
    """Translated: Write a Python function to count the total number of unset bits from 1 to n."""
    """Test Case : assert count_Unset_Bits(2) == 1"""
```
> Solution
```python
def count_Unset_Bits(n):
    cnt = 0;
    for i in range(1,n + 1):
        temp = i;
        while (temp):
            if (temp % 2 == 0):
                cnt += 1;
                temp = temp // 2;
    return cnt;  

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"
    
def main():
    check(1, count_Unset_Bits(2), 1)
    check(2, count_Unset_Bits(5), 4)
    check(3, count_Unset_Bits(14), 17)
```

>> Example 4:
> Instruction
```python
def sum_of_square(n):
    """দ্বিপদী সহগগুলির বর্গক্ষেত্রের যোগফল খুঁজে বের করার জন্য একটি পাইথন ফাংশন লিখুন।"""
    """Translated: Write a Python function to find the sum of squares of the binomial coefficients."""
    """Test Case : assert sum_of_square(4) == 70"""
> Solution
```python
def factorial(start,end): 
    res = 1 
    for i in range(start,end + 1): 
        res *= i      
    return res
    
def sum_of_square(n): 
   return int(factorial(n + 1, 2 * n)/factorial(1, n))

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"
    
def main():
    check(1, sum_of_square(4), 70)
    check(2, sum_of_square(5), 252)
    check(3, sum_of_square(2), 6)
```

>> Example 5:
> Instruction
```python
def extract_date(url):
    """রেজেক্স ব্যবহার করে একটি ইউআরএল থেকে বছর, মাস এবং তারিখ বের করার জন্য একটি ফাংশন লিখুন।"""
    """Translated: Write a function to extract the year, month, and date from a URL using regex."""
    """Test Case : assert extract_date("https://www.washingtonpost.com/news/football-insider/wp/2016/09/02/odell-beckhams-fame-rests-on-one-stupid-little-ball-josh-norman-tells-author/") == [('2016', '09', '02')]"""
```
> Solution
```python
import re
def extract_date(url):
    return re.findall(r'/(\\d{4})/(\\d{1,2})/(\\d{1,2})/', url)

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, extract_date("https://www.washingtonpost.com/news/football-insider/wp/2016/09/02/odell-beckhams-fame-rests-on-one-stupid-little-ball-josh-norman-tells-author/"), [('2016', '09', '02')])
    check(2, extract_date("https://www.indiatoday.in/movies/celebrities/story/wp/2020/11/03/odeof-sushant-singh-rajput-s-death-his-brother-in-law-shares-advice-for-fans-1749646/"), [('2020', '11', '03')])
    check(3, extract_date("https://economictimes.indiatimes.com/news/economy/2020/12/29/finance/pension-assets-under-pfrda-touch-rs-5-32-lakh-crore/articleshow/79736619.cms"), [('2020', '12', '29')])
```
'''

SYSTEM_PROMPT = '''
You are a Python programming assistant. 

The user will provide a function stub where the original docstring is written in Bangla with a translated version and a unit test case.
Your task is to read the Bangla + English (Translated) docstring and the unit test case, understand the requirement, function parameters, return type, and complete the function implementation in Python. 
Your response must be in English, not Bangla, and must only contain valid Python code. 
Do not add explanations, comments, or extra text. Just return the code solution.
Your main task is to carefully read the Bangla + English (Translated) docstring and the unit test case and infer:
1. The expected number of parameter and their types
2. The expected return type
3. The correct and short implementation logic
Important guidelines:
1. The function signature is already provided in the instruction. Implement the function as specified.
2. Include a **main function** (using `def main:`) in your code that contains necessary unit tests or example calls to validate your function.
3. Do **not** call `main()` anywhere in your code. This will be executed externally.
4. Ensure that the function has the parameter format needed to pass the provided unit test.
5. Ensure that the function passes the provided unit tests.If the unit test indicates some class structure give code implementing the class too.
6. Try to keep the code as small and simple as possible.
7. Do **not** change the user provided unit test and make sure the code passes the user provided unit test at all times.
8. If the provided function name does not match the function name in the user provided unit test, use the name from the unit test. 
9. Your response should contain only one python block enclosed in a code block like:\n```python\n# your code here\n```.
'''

PROMPT_TEMPLATE = '''
{examples}

>> Your Task
> Instruction
```python
def {function_call}:
    """{instruction}"""
    """Translated: {instruction_en}"""
    """Test Case : {unit_test}"""
```

Now complete the python code for the function '{function_name}' and add a 'main' function with unit tests. You should use the 'check' function for unit tests, which is helpful for debugging.
'''
LAST_FAILED_ATTEMPT = '''
>> Last failed attempt

> Response:
{last_response}

> Error:
{last_error}

>> Suggested Fix:

{fix_instructions}
'''


In [47]:
# import re
# import json
# dev = convert_csv_to_json("test_v1_en_gemini.csv")
# item = dev[0]
# #print(item["test_list"][0])
# pattern = re.compile(r"```python\s+([\s\S]*?)```", re.MULTILINE)
# match = pattern.search(item["instruction_en"])
# if match:
#     function_name = match.group(1)
# function_call=function_name
# print("Function Name: " + function_name)
# print("Function Call : "+ function_call)
# print("Instruction(bn): " + item["instruction"].split("\n")[0].strip())
# print("Instruction(en): " + remove_python_block(item["instruction_en"]))
# #print(len(item["test_list"]))        
# default_messages = [
#             {"role": "system", "content": SYSTEM_PROMPT},
#         ]
# prompt = PROMPT_TEMPLATE.format(
#         instruction=item["instruction"].split("\n")[0].strip(),
#         instruction_en=remove_python_block(item["instruction_en"]).strip(),
#         function_call=function_call,
#         function_name=function_name,
#         examples=EXAMPLES,
#         last_failed_attempt="",
#         unit_test = item["test_list"][0]
#     )
# messages = default_messages + [{"role": "user", "content": prompt}]
# #response = agent.generate(messages)
# response = gemini_prompt(SYSTEM_PROMPT + prompt)
# print(response)

In [48]:
import time
count = 0
success = 0
total_test_count = 0
passed_test_count = 0
total = 0
max_attempt = 10

# Fixed tqdm bar at top
for item in tqdm(dev_set, desc="Generating", position=0):
    # open folder with task_id
    task_folder = Path(f"./test_results/{model_name}")
    task_folder.mkdir(parents=True, exist_ok=True)
    
    # create a submission.json file if doesn't exist
    if not task_folder.joinpath("submission.json").exists():
        with open(task_folder/"submission.json", "w", newline="", encoding="utf-8") as f:
            json.dump([], f, ensure_ascii=False)

    with open(task_folder/"submission.json", "r", encoding="utf-8") as f:
        submission_data = json.load(f)

    if any(submission["id"] == item["id"] for submission in submission_data):
        print(f"Skipping {item['id']} as it already exists in submission.json")
        matching_submission = next((submission for submission in submission_data if submission["id"] == item["id"]), None)
        flag = (matching_submission["score"] == 1.0)
        if flag:
            # success += flag
            # total += 1
            continue
        
    # prompt = item["instruction"].replace("Exammple", "Function call will be like following")
    last_error = None
    response = None
    fix_instructions = None
    attempt = 0
    history = []
    while True:
        pattern = re.compile(r"```python\s+([\s\S]*?)```", re.MULTILINE)
        match = pattern.search(item["instruction_en"])
        if match:
            function_name = match.group(1)
        function_call=function_name
        #print("Function Name: " + function_name)
        #print("Function Call : "+ function_call)
        #print("Instruction: " + remove_python_block(item["instruction_en"]))

        prompt = PROMPT_TEMPLATE.format(
            instruction=item["instruction"].split("\n")[0].strip(),
            instruction_en=remove_python_block(item["instruction_en"]).strip(),
            function_call=function_call,
            function_name=function_name,
            examples=EXAMPLES,
            unit_test = item["test_list"][0]
            # examples=""
        )
        if last_error is not None:
            if fix_instructions != "Rename the function to the provided function name.Please ensure your code is enclosed in a code block like:\n```python\n# your code here\n":
                prompt += "\n" + LAST_FAILED_ATTEMPT.format(
                    last_response=response,
                    last_error=last_error,
                    fix_instructions=fix_instructions
                )
            else:
                prompt += "\n" + LAST_FAILED_ATTEMPT.format(
                    last_response="",
                    last_error=last_error,
                    fix_instructions=fix_instructions
                )
        # default_messages = [
        #     {"role": "system", "content": SYSTEM_PROMPT},
        # ]
        print(f"======================== {item['id']}.{attempt} =========================")
        # # print(prompt)
        # messages = default_messages + [{"role": "user", "content": prompt}]
            
        # response = agent.generate(messages)
        response = gemini_prompt(SYSTEM_PROMPT + prompt)
        time.sleep(5)
        history.append({"role": "assistant", "content": response})
        
        # Extract Python code block
        pattern = re.compile(r"```python\s+([\s\S]*?)```", re.MULTILINE)
        match = pattern.search(response)
        record = None
        if match:
            code_inside = match.group(1)
            response = "```python\n" + code_inside + "\n```"
            if function_name in code_inside:
                print(f"{function_name} function exists")
                pattern = r'if __name__ == ["\']__main__["\']:\n(?:[ \t]+.*\n?)*'
                clean_code = re.sub(pattern, '', code_inside, flags=re.MULTILINE)
                record = {
                    "id": item["id"],
                    "response": clean_code
                }    
                if re.search(r"def\s+main\s*\(\s*\)", code_inside):
                    print("Main function exists. Processing code.")
                    code_inside = clean_code + "\n\n# Call main function for testing\nmain()"
    
                print(code_inside)
                try:
                    run_code(code_inside)
                    break
                except AssertionError as e:
                    attempt += 1
                    last_error = e
                    fix_instructions = get_fix_instructions(last_error)
                    print(f"Error: {e}")
                    if attempt > max_attempt:
                        break
                except SyntaxError as e:
                    attempt += 1
                    last_error = e
                    fix_instructions = get_fix_instructions(last_error)
                    print(f"Error: {e}")
                    if attempt > max_attempt:
                        break
                except Exception as e:
                    attempt += 1
                    last_error = e
                    fix_instructions = get_fix_instructions(last_error)
                    print(f"Error: {e}")
                    if attempt > max_attempt:
                        break
            else:
                print(f"No {function_name} function exists")
                last_error = "No '"+function_name+"' function found."
                fix_instructions="Rename the function to the provided function name.Please ensure your code is enclosed in a code block like:\n```python\n# your code here\n.Make sure to think less and give code in first 1000 tokens"
                attempt +=1
                if attempt > max_attempt:
                        break
                continue  
        else:
            last_error = "No Python code block found. Please ensure your code is enclosed in a code block like:\n```python\n# your code here\n```"
            fix_instructions="Rename the function to the provided function name.Please ensure your code is enclosed in a code block like:\n```python\n# your code here\n.Make sure to think less and give code in first 1000 tokens"
            print("No python block")

        history.append({"role": "user", "content": f"{last_error}"})
        with open(task_folder/f"{item['id']}.json", "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=4)

    with open(task_folder/f"{item['id']}.json", "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=4)
        
    total += 1
    total_test_count += len(item["test_list"])
    if record is not None:
        responses.append(record)
        count = evaluate_solution(record["response"], item["test_list"])
        success += (count == len(item["test_list"]))
        passed_test_count += count
        # add result to a submission.json
        index = next((i for i, submission in enumerate(submission_data) if submission["id"] == item["id"]), None)
        if index is None:
            submission_data.append({"id": item["id"], "response": record["response"], "score": count/len(item["test_list"])})
        else:
            submission_data[index] = {"id": item["id"], "response": record["response"], "score": count/len(item["test_list"])}
            
        with open(task_folder/"submission.json", "w", encoding="utf-8") as f:
            json.dump(submission_data, f, ensure_ascii=False, indent=4)

    # Clear previous logs and print updated stats
    # clear_output(wait=True)
    print(f"Task: {item['id']} -> Passed {count}/{len(item['test_list'])}")
    print(f"Complete: {success/total*100:.2f}%")
    print(f"Partial: {passed_test_count/total_test_count*100:.2f}%")

Generating:   0%|          | 0/500 [00:00<?, ?it/s]

Skipping 1 as it already exists in submission.json
Skipping 2 as it already exists in submission.json
Skipping 3 as it already exists in submission.json
Skipping 4 as it already exists in submission.json
Skipping 5 as it already exists in submission.json
Skipping 6 as it already exists in submission.json
Skipping 7 as it already exists in submission.json
Skipping 8 as it already exists in submission.json
Skipping 9 as it already exists in submission.json
Skipping 10 as it already exists in submission.json
Skipping 11 as it already exists in submission.json
Skipping 12 as it already exists in submission.json
Skipping 13 as it already exists in submission.json
======================== 13.0 =========================
def maximum_Sum(list1: list[list[int]]) -> int:
 function exists
Main function exists. Processing code.
def maximum_Sum(list1: list[list[int]]) -> int:
    max_sum = 0
    for sublist in list1:
        max_sum += max(sublist)
    return max_sum

def check(test_id, test_val, ex

Generating:   3%|▎         | 13/500 [01:26<54:13,  6.68s/it]

def maximum_Sum(list1: list[list[int]]) -> int:
 function exists
Main function exists. Processing code.
def maximum_Sum(list1: list[list[int]]) -> int:
    total_sum = 0
    for sublist in list1:
        total_sum += max(sublist)
    return total_sum

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"
    
def main():
    check(1, maximum_Sum([[1,2,3],[4,5,6],[10,11,12],[7,8,9]]), 33)
    check(2, maximum_Sum([[1,2],[3,4],[5,6]]), 12)
    check(3, maximum_Sum([[10,20],[30,40],[50,60]]), 150)


# Call main function for testing
main()
Error: Assertion failed: Test 1: Expected 33, got 30
❌ Test 1 failed: assert maximum_Sum([[1,2,3],[4,5,6],[10,11,12],[7,8,9]]) == 33
Task: 13 -> Passed 0/1
Complete: 0.00%
Partial: 0.00%
Skipping 14 as it already exists in submission.json
Skipping 15 as it already exists in submission.json
Skipping 16 as it already exists in submission.json
Skipping 17 as it already exists in submi

Generating:   4%|▍         | 19/500 [02:53<1:18:05,  9.74s/it]

...

def find_char_long(text: str) -> list[str]:
 function exists
Main function exists. Processing code.
def find_char_long(text: str) -> list[str]:
    return None

def square_nums(nums: list[int]) -> list[int]:
    squared_numbers = list(map(lambda x: x**2, nums))
    return squared_numbers

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"
    
def main():
    check(1, square_nums([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]), [1, 4, 9, 16, 25, 36, 49, 64, 81, 100])
    check(2, square_nums([2, 4, 6, 8]), [4, 16, 36, 64])
    check(3, square_nums([1, 3, 5, 7, 9]), [1, 9, 25, 49, 81])


# Call main function for testing
main()
Task: 498 -> Passed 1/1
Complete: 6.45%
Partial: 6.45%
======================== 499.0 =========================


Generating: 100%|█████████▉| 499/500 [45:12<00:08,  8.75s/it]

def find_Rotations(str: str) -> int:
 function exists
Main function exists. Processing code.
def find_Rotations(str: str) -> int:
    temp = str + str
    n = len(str)
    for i in range(1, n + 1):
        if (temp[i: i + n] == str):
            return i
    return n

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, find_Rotations("aaaa"), 1)
    check(2, find_Rotations("geeksforgeeks"), 13)
    check(3, find_Rotations("abab"), 2)


# Call main function for testing
main()
Task: 499 -> Passed 1/1
Complete: 7.94%
Partial: 7.94%
======================== 500.0 =========================


Generating: 100%|██████████| 500/500 [45:20<00:00,  5.44s/it]

def small_nnum(list1: list, n: int) -> list:
 function exists
Main function exists. Processing code.
def small_nnum(list1: list, n: int) -> list:
    list1.sort()
    return list1[:n]

def check(test_id, test_val, expected):
    assert test_val == expected, f"Test {test_id}: Expected {expected}, got {test_val}"

def main():
    check(1, small_nnum([10, 20, 50, 70, 90, 20, 50, 40, 60, 80, 100],2),[10,20])
    check(2, small_nnum([5, 1, 4, 2, 8], 3), [1, 2, 4])
    check(3, small_nnum([5, 2, 9, 1, 5, 6], 4), [1, 2, 5, 5])


# Call main function for testing
main()
Task: 500 -> Passed 1/1
Complete: 9.38%
Partial: 9.38%


In [49]:
print(f"Accuracy: {success/total*100:.2f}%")

import json
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"submission_{timestamp}.json"

with open(filename, 'w', encoding='utf-8') as f:
    json.dump(responses, f, ensure_ascii=False, indent=2)

print(f"Submission file: {filename}")

Accuracy: 9.38%
Submission file: submission_20250910_163704.json


In [50]:
from pathlib import Path
import json
from tqdm import tqdm

dev_set = convert_csv_to_json("test_v1_en_gemini.csv")
task_folder = Path(f"./test_results/{model_name}")
with open(task_folder/"submission.json", "r", encoding="utf-8") as f:
    submission_data = json.load(f)

count = 0
success = 0
total_test_count = 0
passed_test_count = 0
total = 0

for item in tqdm(dev_set, desc="Evaluating", position=0):
    # Find the json with submission["id"] == item["id"]
    matching_submission = next((submission for submission in submission_data if submission["id"] == item["id"]), None)
    if matching_submission:
        print(f"Evaluating {item['id']}...")
        count = evaluate_solution(matching_submission["response"], item["test_list"])
        if count == len(item["test_list"]):
            print(f"✅ All tests passed for {item['id']}")
        success += (count == len(item["test_list"]))
        total_test_count += len(item["test_list"])
        passed_test_count += count
        total += 1

print(f"Accuracy (Pass@1): {success/total*100:.2f}%")
print(f"Unit Test Success: {passed_test_count/total_test_count*100:.2f}%")

Evaluating:  56%|█████▌    | 281/500 [00:00<00:00, 2801.77it/s]

Evaluating 1...
✅ All tests passed for 1
Evaluating 2...
✅ All tests passed for 2
Evaluating 3...
✅ All tests passed for 3
Evaluating 4...
✅ All tests passed for 4
Evaluating 5...
✅ All tests passed for 5
Evaluating 6...
✅ All tests passed for 6
Evaluating 7...
✅ All tests passed for 7
Evaluating 8...
✅ All tests passed for 8
Evaluating 9...
✅ All tests passed for 9
Evaluating 10...
✅ All tests passed for 10
Evaluating 11...
✅ All tests passed for 11
Evaluating 12...
✅ All tests passed for 12
Evaluating 13...
❌ Test 1 failed: assert maximum_Sum([[1,2,3],[4,5,6],[10,11,12],[7,8,9]]) == 33
Evaluating 14...
✅ All tests passed for 14
Evaluating 15...
✅ All tests passed for 15
Evaluating 16...
✅ All tests passed for 16
Evaluating 17...
✅ All tests passed for 17
Evaluating 18...
✅ All tests passed for 18
Evaluating 20...
✅ All tests passed for 20
Evaluating 21...
❌ Test 1 failed: assert func([[1, 2, 6], [1, 3, 4, 5, 7, 8], [1, 3, 5, 6, 8, 9], [2, 5, 7, 11], [1, 4, 7, 8, 12]],3)==[5, 7, 1]
Ev

Evaluating: 100%|██████████| 500/500 [00:00<00:00, 2264.68it/s]

✅ All tests passed for 354
Evaluating 355...
✅ All tests passed for 355
Evaluating 356...
✅ All tests passed for 356
Evaluating 357...
⚠️ Exception in test 1: name 'root' is not defined
Evaluating 358...
✅ All tests passed for 358
Evaluating 359...
✅ All tests passed for 359
Evaluating 360...
✅ All tests passed for 360
Evaluating 361...
✅ All tests passed for 361
Evaluating 362...
✅ All tests passed for 362
Evaluating 363...
✅ All tests passed for 363
Evaluating 364...
✅ All tests passed for 364
Evaluating 365...
✅ All tests passed for 365
Evaluating 366...
✅ All tests passed for 366
Evaluating 367...
✅ All tests passed for 367
Evaluating 368...
✅ All tests passed for 368
Evaluating 369...
✅ All tests passed for 369
Evaluating 370...
✅ All tests passed for 370
Evaluating 371...
✅ All tests passed for 371
Evaluating 372...
✅ All tests passed for 372
Evaluating 373...
❌ Test 1 failed: assert even_bit_toggle_number(10) == 15
Evaluating 374...
✅ All tests passed for 374
Evaluating 375...
✅

In [53]:
import json

# Read the original JSON file
with open("test_results/Gemini_FLash_2.0/submission.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Remove the "score" field from each item
cleaned_data = []
for item in data:
    # keep only id and response
    cleaned_item = {k: v for k, v in item.items() if k != "score"}
    cleaned_data.append(cleaned_item)

# Write to a new JSON file
with open("submission.json", "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=2)
